## Setup

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

## Q1. How many lesson pages

In [11]:
len(files)

72

In [12]:
documents = [raw_repository_file.__dict__ for raw_repository_file in files] 

## Q2. Indexing and searching


In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

def search(question):

    return index.search(
        question,
        num_results=1
    )

search_results = search(question)
search_results

[{'filename': '01-agentic-rag/lessons/14-agentic-loop.md',
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent

## Q3. RAG

In [8]:
from dotenv import load_dotenv

load_dotenv()


from rag_helper import RAGHomeworkBase
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"), base_url="https://opencode.ai/zen/v1/"
)

assistant = RAGHomeworkBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
answer

ModelResponse(output_text='Based on the context, the agentic loop keeps calling the model until it stops by using a `while True` loop that continues as long as the model\'s response contains function calls. The loop checks a `has_function_calls` flag after processing each response. If no function calls are present (i.e., `has_function_calls` is `False`), the loop breaks and stops calling the model.\n\nThe key exit condition is: "No function calls this turn means we\'re done." (from the "The full agent loop" section). The loop relies on the model deciding when it has enough information to provide a final answer without needing further tool calls.', usage=ResponseUsage(input_tokens=7235, input_tokens_details=InputTokensDetails(cached_tokens=7232), output_tokens=384, output_tokens_details=None, total_tokens=7619))

In [6]:
answer.usage

ResponseUsage(input_tokens=7235, input_tokens_details=InputTokensDetails(cached_tokens=7232), output_tokens=475, output_tokens_details=None, total_tokens=7710)

## Q4. Chunking

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

## Q5. RAG with chunking

In [ ]:
index.fit(chunks)
assistant = RAGHomeworkBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
answer

# input_tokens=2336 approximately 3× fewer

ModelResponse(output_text="The agentic loop operates by continuously interacting with the model in a `while` loop, executing any requested tool calls (functions) and returning the results to the model. The loop stops when the model's response **does not contain any function calls**, indicating it has produced a final answer.  \n\nHere’s how it works, based on the provided context:\n\n1. **Continuous Execution**: The loop runs indefinitely (`while True`) and, in each iteration, sends the current conversation history (messages) to the model.\n\n2. **Checking for Function Calls**: After receiving the model’s response, the loop iterates through each item in the output. If an item is a `function_call` (a tool request), the code executes the corresponding function (e.g., a search), appends the result back to the message history, and sets a flag (`has_function_calls = True`).\n\n3. **Exit Condition**: After processing all items, if `has_function_calls` is `False`—meaning the model did not req

## Q6. Turning it into an agent

In [28]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )



agent_tools = Tools()
agent_tools.add_tool(search)

In [29]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(
        model="mimo-v2.5-free",
        client=OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"), base_url="https://opencode.ai/zen/v1/"
        ),
    ),
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)
# 5 with mimo-v2.5-free

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Feature,Plain RAG,Agentic Loop
Control flow,Fixed pipeline,LLM-controlled while loop
Search decisions,Always searches once,Model decides if/when/how to search
Self-correction,❌ No,✅ Yes
Number of LLM calls,1 call,Multiple calls (model-controlled)
Cost,Predictable,Variable (each iteration costs tokens)
Use case,Simple Q&A with known data,Complex tasks requiring reasoning


/workspaces/llm-zoomcamp-2026/01/rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'mimo-v2.5-free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(
